**Github Repo**: https://github.com/zakpayne8283/BaseballAttendance
**Google Collab**: https://colab.research.google.com/github/zakpayne8283/BaseballAttendance/blob/main/AttendanceDashboard.ipynb

# Baseball Attendance

Like many people interested in statistics and data, my favorite sport is baseball. Baseball yields an incredible amount of data and analytics, but for this demonstration I'd like to focus on something very approchable in baseball: attendance data. Starting in 2000, following the implementation of a revenue sharing system, Major League Baseball (MLB) began keeping detailed statistics of the number of tickets distributed for each game played. 

The official data provided by MLB is only numeric and can only provide high level summary data. For example, we can evaluate the 2025 Detroit Tigers data and get some decent summary stats:

| stat | value |
| ---- | ----- |
| mean | 29,796 |
| std dev | 7,587 |
| min | 14,132 |
| median | 30,051 |
| max | 44,735 |

but these stats don't tell us the full story. Sports media often likes to drive the idea of "storylines" throughout a season, and I think attendance helps define that story each season too. Just looking at the average, minimum, or maximum attendance for a team in a season doesn't really tell us anything about why or when people were attending games; combining this data with other periphreal information and plotting it on different types of charts can give us new insight into what happened during a season and what drove MLB game attendance.

Particularly, I think there are a few things that can be very interesting when looking at the attendance data for MLB games that can't observe from just the raw numbers:
- Which opponents drew the largest crowds for the home team? Was it divisional rivals, regional opponents, or big market out of state teams?
- Are fans attending a game for a particular team more on certain days, like weekends? Are they showing up more for day or night games?
- If a team is playing important games (that may impact their playoff chances), will they draw a larger crowd?

## The Data

Baseball attendance data is available in a couple different places but for simplicity I'm going to scrape the data directly from [Baseball Reference](https://www.baseball-reference.com/) instead.

[Here's a sample](https://www.baseball-reference.com/teams/WSN/2019-schedule-scores.shtml) of what a team's schedule looks like on Baseball Reference. There are several columns that will give more insight into attendance trends:

- `Date`
- `Tm`: The team the data was scraped from
- `Unnamed: 4`: `NaN` or `@` to denote the home team
- `Opp`: The opposing team
- `D/N`: Denote a day or night game
- `Attendance`: The number of tickets officially distributed
- `cLI`: Championship Leverage Index, metric for how important the game is in order to win the championship ([further reading](https://www.sports-reference.com/blog/2020/09/__trashed-2/))

### Limitations

The main limitation with this scraped data is that it does not specify the venue and attendance numbers can be skewed in rare instances. For example, the Cincinnati Reds were the home team in the inagural [MLB Speedway Classic](https://en.wikipedia.org/wiki/MLB_Speedway_Classic), which drew a crowd of ~91,000which produces an outlier in the results. Luckily, occurences like this are rare and should generally have a minimal impact on the data.

### Baseball Reference

Baseball Reference is a great place to view data for baseball, however it can be a little challenging downloading data from them. They don't have any public datasets or an API available, so data needs to be scraped directly from their site, which can be tedious because they rate limit requests to about 20 requests per minute.

### Loading the Data

I've included the full raw data in this repo (`raw_output.csv`) so you don't need to wait for the scraping to run, but if you want to download the data yourself you can run the cell below to fetch all of the data (**NOTE**: this does not work in Google Colab since the helper script is not available). To minimize waiting time, you can also adjust the `start_year` and `end_year`, to focus only on the years you'd like to view. Running the scraping script for all years (2000-2025) takes about 75 minutes.

To summarize: this script iterates over a list of teams and years, and collects each team's schedule for that year, writing the data to `raw_output.csv`.

In [ ]:
from data_helper import DataHelper
DataHelper().collect_data()
# DataHelper(start_year=2025, end_year=2025).collect_data()

### Load the Data

In [24]:
import pandas as pd

# df = pd.read_csv('raw_output.csv') # local run only
df = pd.read_csv('https://raw.githubusercontent.com/zakpayne8283/BaseballAttendance/refs/heads/main/raw_output.csv')

### Cleaning the Data

First, there are several unneeded columns that can be removed:

In [25]:
df = df.drop(columns=['Gm#', 'W/L', 'W-L', 'R', 'RA', 'Inn', 'Rank', 'GB', 'Win', 'Loss', 'Save', 'Time', 'Streak', 'Orig. Scheduled'])

Next, there are two specific row-by-row cases that need to be cleaned:

 First, by default Baseball Reference repeats its column headers on tables at each new month. This makes it easier to view the data in a web browser, but it needs to be filtered out of this data.
 
 Second is missing data for teams with cancelled/relocated seasons. For example, the 2025 "Sacramento" Athletics played [this schedule](https://www.baseball-reference.com/teams/ATH/2025-schedule-scores.shtml), but were originally supposed to play [this schedule](https://www.baseball-reference.com/teams/OAK/2025-schedule-scores.shtml) before they relocated from Oakland to Sacramento [on their way to Las Vegas](https://en.wikipedia.org/wiki/Oakland_Athletics_relocation_to_Las_Vegas).
 
 We can filter out both of these cases by only selecting for rows where `Unnamed: 2` is "boxscore", representing a game which was played and completed.

In [26]:
df = df[df['Unnamed: 2'] == 'boxscore'].drop(columns=['Unnamed: 2'])

Next, there are a lot of duplicate rows in the data set. Because of the way the data was pulled, an entry is made for both teams for each game. For example:

| Date | Tm | Unnamed: 4 | Opp | ... |
| ---- | -- | ---------- | --- | --- |
| Tuesday, Apr 4 | ARI | NaN | PHI | ... |
| Tuesday, Apr 4 | PHI | @ | ARI | ... |

For the Philidelphia Phillies @ Arizona Diamondbacks game on April 4, 2000.

Fixing this requires only accepting rows where `Unnamed: 4` is `NaN`, essentially selecting only home games for `Tm`:

In [27]:
df = df[df['Unnamed: 4'].isna()].drop(columns=['Unnamed: 4'])

At this point, the average number of games each team plays should be approximately 81. Sometimes this number will vary, based on cancelled games or relocated games, but in a typical season this will be exactly 81, like in 2025:

In [28]:
print(f'Home games per team: {len(df[df['year'] == 2025]) / 30}')

Home games per team: 81.0


Next, the `Date` field should be stored as a pandas datetime field for graphing purposes later:

In [29]:
# strip any double header information from the raw date string (example: Thursday, Apr 4 (1))
df['Date'] = df['Date'].str.replace(r'\s*\(\d+\)$', '', regex=True)
df['date'] = pd.to_datetime(df['Date'] + ' ' + df['year'].astype(str), format='%A, %b %d %Y')
df = df.drop(columns=['Date'])

There's one last scenario to account for: games with no attendance. This can happen for several reasons, but most notably occured during the COVID-19 pandemic. Filtering out rows with no attendance will solve that:

In [30]:
df = df[df['Attendance'].notna()]

Renaming the columns to something more readable and correcting the data types will also be helpful:

In [31]:
df = df.rename(columns={
    'Tm': 'home_team',
    'Opp': 'away_team',
    'D/N': 'day_night',
    'Attendance': 'attendance',
})

df = df.astype({
    'attendance': 'int32',
    'cLI': 'float32'
})

And lastly, we need to assign a game number for each game, denoting which home game it is for that team that season. Since sometimes teams go weeks at a time without playing home games, an x-axis represented by a date can be misleading, since some months will have more games.

In [32]:
df = df.sort_values('date')
df['game_number'] = df.groupby(['home_team', 'year']).cumcount() + 1

Lastly, let's save the processed dataframe to easily refer to later, if we don't want to pull and clean all the data again.

In [33]:
df.to_csv('processed_data.csv')

## Graphing with Plotly and Plotly Dash (Visualization Library)

For this project, I chose [Plotly](https://plotly.com/) and their dashboard/app tool, [Plotly Dash](https://plotly.com/dash/). Plotly is a very powerful charts library - created by a company of the same name - that's built on Javascript to create highly interactive charts and displays. It supports a wide variety of charts and graphs out of the box and allows for high levels of customization. Plotly Dash is built on top of Plotly, combining a small local webserver (Flask), the Javascript implementation of Plotly, and React.js. Dash allows you to map your python elements directly to HTML, creating a simple web app without the need to do any HTML or Javascript yourself.

Plotly also integrates directly into Jupyter with no extensions needed, making it a great addition to any notebook utilizing data visualization. It includes additional Jupyter configuration options like multiple rendering types and themes natively.

Personally, I chose Plotly because I had a small amount of previous experience building simple charts with it before and it's been recommended to me in the past by coworkers who were former data analysts. While working with it, I found several aspects of the library that I felt made it a good fit for this project. First is that it's quite easy to spin up charts using Plotly and its depth of documentation makes any troubleshooting easier due to its well of public information. Secondly, I think the HTML-styled dashboarding is a very intuitive way of building the dashboard itself; I spent very little time putting down boilerplate code because it matches HTML's simplicity. Then when it came to styling, my familiarity with CSS made it easy to style the charts as well.  

The major drawback I identified with it was the configuration options needed to work with Google Colab. Since it's heavily reliant on Javascript and local webservers, the configuration differences between a local notebook and Google Colab and differ quite a bit. For this notebook to run in Google Colab, you must add `app.run(mode="inline")` to any of the `app.run()` Plotly calls.

### Plotly Examples

Here's a quick example of a histogram in Plotly using randomly generated data:

In [34]:
import numpy as np
import plotly.express as px

data = np.random.normal(loc=50, scale=10, size=1000)
fig = px.histogram(data, nbins=30, labels={'value': 'Random Value'}, title='Quick Plotly Demo: Histogram of Random Data')
fig.update_layout(showlegend=False, title_x=0.5)
fig.show()

as you can see, you can hover over the elements and have an interactive chart, no extra work required. 

If you want to incorporate more into the display, you can add python blocks that correspond to HTML elements:

In [35]:

from dash import Dash, html, dcc
import plotly.express as px
import numpy as np

# create a new Flask application
app = Dash()

# randomly generate and plot some data
data = np.random.normal(loc=50, scale=10, size=1000)
fig = px.histogram(data, nbins=30, labels={'value': 'Random Value'})
fig.update_layout(showlegend=False, title_x=0.5)

# create an HTML <div> that contains the title in an <h1>
app.layout = html.Div([
    html.H1('Chart Title'),
    html.P('This is a paragraph about the interactive histogram below.'),
    dcc.Graph(figure=fig),
], style={'background-color': 'white', 'text-align': 'center', 'font-family': 'serif', 'padding': '20px'})

# run the Flask application
app.run(mode="inline")

As you can see, the `html.Div({})` object creates a `<div>` within the Flask application and even let's us use CSS style attributes to customize it.

### Installation, Source, and Licensing

Plotly and Dash can be installed using pip:

```
pip install plotly
pip install dash
```

The projects are both open source and have MIT (Open Source) licensing. It is worth noting that the company Plotly is a privately held, VC backed company and they do offer cloud and enterprise products that are related to Plotly, however both Plotly and Dash are completely free and open source ([Plotly Source](https://github.com/plotly/plotly.py) | [Dash Source](https://github.com/plotly/dash))

## Plotting MLB Attendance Data (Visualization Technique)

Now let's begin plotting the MLB Attendance data. I think there are four charts that can both highlight the power of Plotly/Dash and give us more insight into MLB attendance. Let's start by loading the data as needed and all the Plotly/Dash objects, along with some helper data I wrote.

In [37]:
from dash import Dash, html, dcc, callback, Output, Input
import plotly.express as px
import pandas as pd
from team_helper import get_team_colors, TEAM_NAMES

# load the data if starting from here
try:
    df == None
except NameError:
    # df = pd.read_csv('processed_data.csv')
    df = pd.read_csv('https://raw.githubusercontent.com/zakpayne8283/BaseballAttendance/refs/heads/main/processed_data.csv')

I'm also going to set some hard-coded values to just demonstrate each graph first. Later, these values will be dynamic in the interactive dashboard:

In [38]:
# which team's data to display, the Detroit Tigers by default
team = 'DET'
# 2025 season data
year = int(df['year'].max())

# filtered down data frame focusing on the team & year
team_df = df[(df['home_team'] == team) & (df['year'] == year)]

I'm also defining a short helper script that will enable matching a team code to that team's primary color or full team name:

In [44]:
TEAM_COLORS = {'ANA': '#003263', 'ARI': '#A71930', 'ATL': '#CE1141', 'BAL': '#DF4601', 'BOS': '#BD3039', 'CHC': '#0E3386', 'CHW': '#27251F', 
               'CIN': '#C6011F', 'CLE': '#00385D', 'COL': '#333366', 'DET': '#0C2340', 'FLA': '#009CA6', 'HOU': '#002D62', 'KCR': '#004687',
               'LAA': '#003263', 'LAD': '#005A9C', 'MIA': '#00A3E0', 'MIL': '#12284B', 'MIN': '#002B5C', 'MON': '#003087', 'NYM': '#002D72', 
               'NYY': '#003087', 'ATH': '#003831', 'OAK': '#003831', 'PHI': '#E81828', 'PIT': '#27251F', 'SDP': '#2F241D', 'SFG': '#FD5A1E', 
               'SEA': '#0C2C56', 'STL': '#C41E3A', 'TBR': '#092C5C', 'TBD': '#092C5C', 'TEX': '#003278', 'TOR': '#134A8E', 'WSN': '#AB0003'
}

def get_team_colors(teams):
    return {team: TEAM_COLORS.get(team, '#888888') for team in teams}

TEAM_NAMES = { 'ANA': 'Los Angeles Angels of Anaheim', 'ARI': 'Arizona Diamondbacks', 'ATL': 'Atlanta Braves', 'BAL': 'Baltimore Orioles',
               'BOS': 'Boston Red Sox', 'CHC': 'Chicago Cubs', 'CHW': 'Chicago White Sox', 'CIN': 'Cincinnati Reds', 'CLE': 'Cleveland Guardians',
               'COL': 'Colorado Rockies', 'DET': 'Detroit Tigers', 'FLA': 'Floria Marlins', 'HOU': 'Houston Astros', 'KCR': 'Kansas City Royals',
               'LAA': 'Los Angeles Angels', 'LAD': 'Los Angeles Dodgers', 'MIA': 'Miami Marlins', 'MIL': 'Milwaukee Brewers', 'MIN': 'Minnesota Twins',
               'NYM': 'New York Mets', 'NYY': 'New York Yankees', 'ATH': 'Sacramento Athletics', 'OAK': 'Oakland Athletics', 'PHI': 'Philadelphia Phillies',
               'PIT': 'Pittsburgh Pirates', 'SDP': 'San Diego Padres', 'SFG': 'San Francisco Giants', 'SEA': 'Seattle Mariners', 'STL': 'St. Louis Cardinals',
               'TBD': 'Tampa Bat Devil Rays', 'TBR': 'Tampa Bay Rays', 'TEX': 'Texas Rangers', 'TOR': 'Toronto Blue Jays', 'WSN': 'Washington Nationals',
               'MON': 'Montreal Expos',
}

### Line Chart

A simple line chart is a great place to start both when working with a new charts library and a new data set. Earlier, I mentioned each team plays about 81 home games and that I numbered these using `game_number` to make a uniform distribution of games. The first chart is going to be a simple line chart to look at which home game number it was and what the attendance was.

To create a line chart using plotly, you can use `px.line()` and supply it the dataframe, x, and y values. I've also set the line to adjust its color to be the same as the team's primary color, and set a few labels as well. For more information on what you can do with line charts, refer to the [line chart documentation](https://plotly.com/python/line-charts/).

In [45]:
line_graph = px.line(team_df, x='game_number', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
              labels={'game_number': 'Home Game Number', 'attendance': 'Attendance', 'home_team': 'Team'}, title=f'{team} Attendance by Home Game Number')
# hide legend, center title
line_graph.update_layout(showlegend=False, title_x=0.5)
line_graph.show()

I think a line chart is a great place to start exploring this data because it can raise a lot of great questions about the season at a glance. For example, we see quite a few peaks and valleys throughout the year, what causes those? Are the valleys weekday games and the peaks weekend games? Or possibly are the peaks when highly popular teams and/or players are visiting?

One interesting thing to note in this data straight away is that the minimum attendance value happened early in the season, occuring during the fourth, fifth, and sixth home games. While we could explore these data points using the other information at our disposal, we wouldn't find the true reason anywhere in our data; attendance was low those days (April 7th, 8th, and 9th) because the temperature at game time was 38, 34, and 42 degrees respectively.

### Bar Chart

One thing I wanted to explore was if a popular away team drove attendance up for some teams, and I think a bar chart is a great way to do that. Since teams play each other multiple times each year, we can think of `away_team` as a category to average values across.

To create a bar chart using Plotly, you can use `px.bar()` and provide it the data frame, the x values (the numerical values) and y values (the categorical values). In my case, I specifically wanted a horizontal bar chart, so I set `orientation='h'`. Like the line chart above, I set each bar's color value using a map of the team colors, but set them based on the `away_team` value this time. One small thing I had to do was also use `.update_yaxes()` to force all the labels to appear, since by default Plotly was removing some of them. For more information on the Plotly bar charts, refer to [the documentation](https://plotly.com/python/bar-charts/).

In [46]:
# get the average attendance by away team and sort descending 
dff_grouped = team_df.groupby('away_team').agg(avg_attend=('attendance', 'mean')).reset_index().sort_values('avg_attend', ascending=False)

#plot
bar_chart = px.bar(dff_grouped, y='away_team', x='avg_attend', orientation='h',
                   color='away_team', color_discrete_map=get_team_colors(dff_grouped['away_team']),
                   labels={'away_team': 'Opponent', 'avg_attend': 'Average Attendance'},
                   title='Average Attendance by Opponent')

# force all team labels to appear
bar_chart.update_yaxes(tickmode='linear', dtick=1)
# remove legend and center title
bar_chart.update_layout(showlegend=False, title_x=0.5)
bar_chart.show()

Exploring this bar chart is very interesting! I chose a bar chart for this question because it would make direct comparison of attendance between the away teams very easy, and we're able to clearly see the differences between the teams the Detroit Tigers hosted.

My original question was if popular teams drive attendance, and this provides an interesting view of the answer. In Detroit, it seems that the popular teams didn't drive attendance in 2025 (though the NYY games were those cold April games mentioned above). Instead, attendance was seemlingly driven by regional opponents such as Chicago, Toronto, Cincinnati and Cleveland, though some popular team like Atlanta were in the mix as well.

### Scatter Plot

The scatter plot is a very strong tool for evaluating the relationship of data in this set. In this case, I want to evaluate the relationship between the championship leverage index (how important the game is to winning the championship) and attendance to see if attendance increase in more important games.

To create a scatter plot in Plotly, you can run `px.scatter()` and pass the data frame, the x values, and y values. Like the other charts so far, I'm also specifying certain colors for the dots on the plot. As you can see below I'm also annotating the scatter plot with the correlation coefficient (from pandas) using `.add_annotation()`. For more information, refer to the [scatter plot documentation](https://plotly.com/python/line-and-scatter/).

In [53]:
# plot
scatter_plot = px.scatter(team_df, x='cLI', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
                          labels={'cLI': 'Championship Leverage Index (Game Importance)', 'attendance': 'Attendance'},
                          title='Attendance vs. Game Importance', trendline='ols')

# add correlation
r_value = team_df['cLI'].corr(team_df['attendance'])
scatter_plot.add_annotation(text=f'r = {r_value:.3f}', xref='paper', yref='paper', x=0.98, y=0.98, showarrow=False, font=dict(size=14))

# remove legend and center title
scatter_plot.update_layout(showlegend=False, title_x=0.5)
scatter_plot.show()

Again, since I wanted to directly explore the relationship between attendance and game importance a scatter plot was a natural fit. However, it seems like this graph shows only a loose correlation between the game importance and attendance for the 2025 Tigers. Part of this could be that the Tigers held a large lead through most of the year, and played relatively few highly important games at home.

### Jittered Dot and Box Plot

One of the questions I asked early on when evaluating the line chart was if there was lower attendance on weeknights than on weekend games. I also was wondering if day or night games tended to have more attendance. There tends to be a day of the week pattern of when day or night games are played (Wednesday and Sunday are typically day games), so I thought it would be best to highlight the different game times as well. To answer this I wanted to use a jittered dot plot with a box plot overlay, where the dots were differently colored based on day or night games.

You can create the jittered dot plot using `px.strip()`, and pass the data frame, x and y values. This time I assigned dot colors based on day or night games. More information on jittered dot plots can be found in the [documentation](https://plotly.com/python/strip-charts/).

To create the box plots, run `px.box()` and pass the data frame, x and y values. I wanted to directly control the opacity and line width of my boxplot to prevent it from overpowering the data on the dot plot, so I manually set those using `line=dict(color='black', width=1)` and `opacity=0.4`.

Then to overlay the box plot, I used the following code to set the two different plots to overlay and add each box plot on top.
```
jittered_dot_plot.update_layout(boxmode='overlay', title_x=0.5)
for trace in box_overlay.data:
    jittered_dot_plot.add_trace(trace)
```

In [47]:
# assign a day of the week to each game and establish the order of the week
team_df['day_of_week'] = pd.to_datetime(team_df['date']).dt.day_name()
day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']

# plot the jittered dot
jittered_dot_plot = px.strip(
    team_df, x='day_of_week', y='attendance', color='day_night',
    category_orders={'day_of_week': day_order},
    color_discrete_map={'D': '#F5A623', 'N': '#1B1F3B'},
    labels={'day_of_week': 'Day of Week', 'attendance': 'Attendance', 'day_night': 'Day/Night'},
    title='Attendance by Day of Week and Game Time'
)

# create a box plot too
box_overlay = px.box(team_df, x='day_of_week', y='attendance', category_orders={'day_of_week': day_order})
box_overlay.update_traces(fillcolor='rgba(255,255,255,0)', line=dict(color='black', width=1), opacity=0.4, showlegend=False)

# overlay the box plot on the jittered dot
jittered_dot_plot.update_layout(boxmode='overlay', title_x=0.5)
for trace in box_overlay.data:
    jittered_dot_plot.add_trace(trace)

jittered_dot_plot.show()

My goal was to evaluate if weeknight or weekend games had higher or lower attendance and this graph shows me exactly what I was looking for! The medians values for each weekend day were each higher than the maximums for each week day (except Monday, which gets a boon from Labor Day). The usage of the jittered dot / box plots were a great way to see the distribution of attendance based on the day of the week.

## Plotly Dash Dashboard

Now it's time to put all of these charts together and make them a bit more dynamic. Right now to view data for a different team from a different year, you would need to manually set the team code and year and re-run all of the charts again. 

However, we can use the HTML building blocks above to not only render graphs together, but control inputs using HTML input elements to allow for different teams or years to be selected instead. When looking at the python code in the next cell, imagine the `app.layout` variable to look like the following HTML:
```
<div>
    <h1>Attendance Data by Team and Year</h1>
    <div>
        <div>
            <select>
            <!-- Team Options -->
            ...
            </select>
        </div>
        <div>
            <select>
            <!-- Year Options -->
            ...
            </select>
        </div>
    </div>
    <div>
        <div>
            <graph>
                <!-- Line Chart -->
            </graph>
            <graph>
                <!-- Bar Chart -->
            </graph>
        </div>
        <div>
            <graph>
                <!-- Scatter Plot -->
            </graph>
            <graph>
                <!-- Jittered Dot / Box Plot -->
            </graph>
        </div>
    </div>
</div>
```

In [49]:
app = Dash()

app.layout = html.Div([
    html.H1(children='Attendance Data by Team and Year', style={'textAlign':'center'}),
    html.Div([
        html.Div(
            dcc.Dropdown(
                # get real team names from my helper file
                # HTML for this is like <option value="DET">Detroit Tigers</option>
                [{'label': TEAM_NAMES.get(code, code), 'value': code} for code in sorted(df['home_team'].unique(), key=lambda c: TEAM_NAMES.get(c, c))],
                'DET', id='team-dropdown'
            ),
            style={'width': '70%', 'padding': '10px'}
        ),
        html.Div(
            dcc.Dropdown(sorted(df['year'].unique()), df['year'].max(), id='year-dropdown'),
            style={'width': '30%', 'padding': '10px'}
        ),
    ], style={'display': 'flex', 'flexDirection': 'row'}),

    html.Div([
        html.Div([
            dcc.Graph(id='attendance-by-team-graph', style={'width': '50%'}),
            dcc.Graph(id='attendance-by-opponent-graph', style={'width': '50%'}),
        ], style={'display': 'flex', 'flexDirection': 'row'}),

        html.Div([
            dcc.Graph(id='attendance-by-game-importance-graph', style={'width': '50%'}),
            dcc.Graph(id='attendance-by-day-or-night-graph', style={'width': '50%'}),
        ], style={'display': 'flex', 'flexDirection': 'row'}),
    ], style={'display': 'flex', 'flexDirection': 'column', 'gap': '10px'})
], style={'backgroundColor': '#ffffff', 'color': '#111', 'minHeight': '100vh', 'padding': '20px', 'font-family': 'sans-serif'})

But this code won't make the dashboard dynamic quite yet. This was only the HTML - what makes it dynamic is the Javascript from Plotly.js and React.js to update the values based on our inputs.

To accomplish this, we use a method decorator, `@callback`, to specify which elements in our HTML above are inputs and which ones are outputs. Then the method it's decorating takes those same inputs as parameters, and we add the chart logic we defined above to have it generate charts each time our inputs are updated. Then when one of the inputs is updated, the method annotated as `@callback` is called back to to re-render the charts as needed.

One note, I've also added `.update_xaxes()` or `.update_yaxes()` to the charts as needed to control the global mins/maxs of each chart now. Since we can now change teams quickly, teams should be held to the same scope if they're in the same year.

Lastly, I'm using `app.run(mode="inline")` here to run the dashboard to be compatible with Google Colab, however other methods are available.

In [50]:
@callback(
    Output('attendance-by-team-graph', 'figure'),
    Output('attendance-by-opponent-graph', 'figure'),
    Output('attendance-by-game-importance-graph', 'figure'),
    Output('attendance-by-day-or-night-graph', 'figure'),
    Input('team-dropdown', 'value'),
    Input('year-dropdown', 'value')
)
def update_graph(team, year):
    dff = df[(df['home_team'] == team) & (df['year'] == year)]
    year_df = df[df['year'] == year]
    attend_min, attend_max = year_df['attendance'].min(), year_df['attendance'].max()

    # --- line chart ---
    line_graph = px.line(dff, x='game_number', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
                  labels={'game_number': 'Home Game Number', 'attendance': 'Attendance', 'home_team': 'Team'}, title='Attendance Home Game Number')
    line_graph.update_yaxes(range=[attend_min, attend_max])
    line_graph.update_layout(showlegend=False, title_x=0.5)

    # --- bar chart ---
    dff_grouped = dff.groupby('away_team').agg(avg_attend=('attendance', 'mean')).reset_index()
    dff_grouped = dff_grouped.sort_values('avg_attend', ascending=False)
    bar_chart = px.bar(dff_grouped, y='away_team', x='avg_attend', orientation='h',
                       color='away_team', color_discrete_map=get_team_colors(dff_grouped['away_team']),
                       labels={'away_team': 'Opponent', 'avg_attend': 'Average Attendance'},
                       title='Average Attendance by Opponent')
    max_avg_attendance = year_df.groupby(['home_team', 'away_team'])['attendance'].mean().max()
    bar_chart.update_xaxes(range=[0, max_avg_attendance])
    bar_chart.update_yaxes(tickmode='linear', dtick=1)
    bar_chart.update_layout(showlegend=False, title_x=0.5)

    # --- scatter plot ---
    scatter_plot = px.scatter(dff, x='cLI', y='attendance', color='home_team', color_discrete_map=get_team_colors([team]),
                              labels={'cLI': 'Championship Leverage Index (Game Importance)', 'attendance': 'Attendance'},
                              title='Attendance vs. Game Importance')
    scatter_plot.update_yaxes(range=[attend_min, attend_max])
    scatter_plot.update_layout(showlegend=False, title_x=0.5)

    # --- jittered dot plot + box overlay ---
    dff['day_of_week'] = pd.to_datetime(dff['date']).dt.day_name()
    day_order = ['Sunday', 'Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday']
    jittered_dot_plot = px.strip(
        dff, x='day_of_week', y='attendance', color='day_night',
        category_orders={'day_of_week': day_order},
        color_discrete_map={'D': '#F5A623', 'N': '#1B1F3B'},
        labels={'day_of_week': 'Day of Week', 'attendance': 'Attendance', 'day_night': 'Day/Night'},
        title='Attendance by Day of Week and Game Time'
    )
    box_overlay = px.box(dff, x='day_of_week', y='attendance', category_orders={'day_of_week': day_order})
    box_overlay.update_traces(fillcolor='rgba(255,255,255,0)', line=dict(color='black', width=1), opacity=0.4, showlegend=False)
    jittered_dot_plot.update_layout(boxmode='overlay', title_x=0.5)
    for trace in box_overlay.data:
        jittered_dot_plot.add_trace(trace)

    return line_graph, bar_chart, scatter_plot, jittered_dot_plot

app.run(mode="inline")

### Why This Scope
Combining these four charts together paints a great picture of attendance for any team in the data set. The line chart gives a great starting point for exploration, giving a high level picture of the game-by-game attendance and trends. Of course why certain games have higher or lower attendance is a great question and we can gain insight by looking to our other graphs. The bar chart of average attendance by opponent can fill in some of the gaps from our bar chart of why certain stretchs are up or down based on the opponent and the same could be said for the scatter plot of game importance and the jittered dot / box plot for days of the week. The four views side-by-side can help us see why certain stretchs over or under perform on attendance and complement each other well.

As I was building this dashboard I explored a couple different options, but ultimately chose this layout - a single team/year with these four charts - because they complemented the data the best. Layering multiple teams on the same line graph was interesting, but it presented many issues; not every team plays on the same day or has the same distribution of which day of the week or month they play in their home park. Similarly, different teams play in parks of varying capacities, in cities with different climates, with very different population distributions. Sometimes the same team changes cities or venues year-to-year, which can make yearly comparisons among a single team challenging. All of these factored in to my decision to scope my dashboard selection to just a single team and a single year at a time.